# CloneCast — one-time RVC training (Phase 8.1)

Trains your voice model and packages everything the converter kernel needs.

**Before Run All:** attach dataset `clonecast-voice-raw` (10+ min of your clean voice),
set Accelerator = **GPU T4**, Internet = **ON**.

**Output:** `/kaggle/working/model-dataset/` → save as private dataset `clonecast-rvc-model`.

NOTE (validated during Phase 8.1): the training CLI of the pinned RVC commit is
invoked below the same way its webui does; if a step errors, check the printed
help of that script — the repo layout is train/preprocess.py, train/dataset/extract_f0.py,
train/dataset/extract_hubert_feature.py, train/train.py, train/train_index.py.

In [ ]:
# Cell 1 — pinned environment (same stack as the converter kernel)
RVC_COMMIT = "4338f12c3c28c80b3ac015e2d0df66c41592746d"  # keep in sync with RvcAssets.RVC_COMMIT
EXP_NAME = "clonecast"
SAMPLE_RATE = "40k"   # RVC v2 default
EPOCHS = 300
BATCH = 8

import subprocess, sys, os
def run(cmd): print('>>', cmd); subprocess.run(cmd, shell=True, check=True)

run(f'pip install -q torch==2.7.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu128')
run(f'git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI /kaggle/working/rvc')
run(f'git -C /kaggle/working/rvc checkout {RVC_COMMIT}')
# their py312 requirements minus CN mirror lines, numpy<2 enforced
req = open('/kaggle/working/rvc/requirments_cu128_py312.txt').read().splitlines()
open('/kaggle/working/req.txt','w').write('\n'.join(l for l in req if not l.strip().startswith('--index-url')))
open('/kaggle/working/constraints.txt','w').write('numpy<2\ntorch==2.7.1\ntorchaudio==2.7.1\n')
run('pip install -q -r /kaggle/working/req.txt -c /kaggle/working/constraints.txt')
import torch; print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Cell 2 — runtime assets the converter also needs (downloaded once, shipped in the model dataset)
# hubert_base: transformers-format ContentVec (matches infer/hubert.py HubertModelWithFinalProj)
run('pip install -q huggingface_hub')
from huggingface_hub import snapshot_download, hf_hub_download
hubert_dir = snapshot_download('lengyue233/content-vec-best', local_dir='/kaggle/working/hubert_base')
rmvpe = hf_hub_download('lj1995/VoiceConversionWebUI', 'rmvpe.pt', local_dir='/kaggle/working')
print('hubert_base ->', hubert_dir)
print('rmvpe ->', rmvpe)
# make them visible to the repo for training too
os.makedirs('/kaggle/working/rvc/assets', exist_ok=True)
if not os.path.exists('/kaggle/working/rvc/assets/hubert_base'):
    os.symlink('/kaggle/working/hubert_base', '/kaggle/working/rvc/assets/hubert_base')
os.environ['rmvpe_root'] = '/kaggle/working'

In [ ]:
# Cell 3 — preprocess + features + f0 (paths per the pinned repo's webui invocations)
os.chdir('/kaggle/working/rvc')
sys.path.insert(0, '/kaggle/working/rvc')
VOICE_DIR = '/kaggle/input/clonecast-voice-raw'
EXP_DIR = f'/kaggle/working/rvc/logs/{EXP_NAME}'
os.makedirs(EXP_DIR, exist_ok=True)

# The exact CLI of these steps can differ per commit — print help first, then call.
run(f'python train/preprocess.py --help || true')
run(f'python train/dataset/extract_f0.py --help || true')
run(f'python train/dataset/extract_hubert_feature.py --help || true')

# TODO(8.1): fill the three calls below from the help output / webui train tab.
# Expected shape (classic RVC):
#   preprocess:  input dir, sample rate, n_proc, exp dir
#   f0:          exp dir, method rmvpe
#   features:    exp dir, v2
raise SystemExit('Fill Cell 3 commands from the printed --help output, then re-run from this cell.')

In [ ]:
# Cell 4 — train + build index (fill args like Cell 3, from train/utils.py --help)
run(f'python train/train.py --help || true')
run(f'python train/train_index.py --help || true')
# TODO(8.1): train with EXP_NAME, SAMPLE_RATE, EPOCHS, BATCH, v2, rmvpe; then train_index.

In [ ]:
# Cell 5 — package the model dataset for the converter kernel
import glob, shutil, json
OUT = '/kaggle/working/model-dataset'
os.makedirs(OUT, exist_ok=True)

weights = sorted(glob.glob(f'/kaggle/working/rvc/assets/weights/{EXP_NAME}*.pth')) or \
          sorted(glob.glob(f'{EXP_DIR}/**/*.pth', recursive=True))
indexes = sorted(glob.glob(f'{EXP_DIR}/**/added_*.index', recursive=True))
assert weights, 'No trained .pth found — check Cell 4'
assert indexes, 'No added_*.index found — check Cell 4'
shutil.copy(weights[-1], f'{OUT}/model.pth')
shutil.copy(indexes[-1], f'{OUT}/model.index')
shutil.copytree('/kaggle/working/hubert_base', f'{OUT}/hubert_base', dirs_exist_ok=True)
shutil.copy('/kaggle/working/rmvpe.pt', f'{OUT}/rmvpe.pt')
json.dump({'exp': EXP_NAME, 'sr': SAMPLE_RATE, 'epochs': EPOCHS, 'rvc_commit': RVC_COMMIT},
          open(f'{OUT}/config.json', 'w'), indent=2)
print('Packaged:', os.listdir(OUT))
print('Now: sidebar -> Output -> New Dataset -> slug clonecast-rvc-model (PRIVATE)')